# 1. Setup & Import

In [1]:
import os, sys, json, torch, penman, numpy as np, pandas as pd
from torch.utils.data import Dataset, DataLoader

from transformers import AutoConfig
from model_interface.modeling_bart import MBartForConditionalGeneration
from model_interface.tokenization_bart import AMRBartTokenizer

print(f"Pytorch Version : {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")

c:\D\ITB\riset_gnn\generate_amr\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Pytorch Version : 2.12.0+cu126
CUDA Available : True


# 2. Load Model & Tokenizer (NLLB & Nafkhan)

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Default Device : {device}")

Default Device : cuda


## Model AMR (Nafkhan)

In [3]:
AMR_MODEL_PATH = os.path.join(
    os.getcwd(),
    "..",
    "models",
    "mbart-en-id-smaller-concat-finetuned",
    "mbart-en-id-smaller-concat-finetuned"
)

amr_config = AutoConfig.from_pretrained(AMR_MODEL_PATH)
print(f"Model type : {amr_config.model_type}")
print(f"Model type : {amr_config.architectures}")
print(f"Model type : {amr_config.vocab_size}")

Model type : mbart
Model type : ['MBartForConditionalGeneration']
Model type : 38025


In [4]:
amr_tokenizer = AMRBartTokenizer.from_pretrained(AMR_MODEL_PATH, use_fast=False)
amr_model = MBartForConditionalGeneration.from_pretrained(AMR_MODEL_PATH, config=amr_config)
amr_model = amr_model.to(device)

amr_model.eval()

print(f"Model Loaded on : {amr_model.device}")
print(f"Model Parameters : {sum([p.numel() for p in amr_model.parameters() if p.requires_grad is True])}")


Added 0 AMR tokens
Model Loaded on : cuda:0
Model Parameters : 393761792


## Model NLLB

In [5]:
# from transformers import NllbTokenizer, AutoModelForSeq2SeqLM
# from huggingface_hub import snapshot_download

# # Download model locally first to bypass transformers/huggingface_hub version mismatch
# nllb_local_path = snapshot_download("facebook/nllb-200-distilled-1.3B")
# print(f"NLLB model downloaded to: {nllb_local_path}")

# nllb_tokenizer = NllbTokenizer.from_pretrained(nllb_local_path)
# nllb_model = AutoModelForSeq2SeqLM.from_pretrained(nllb_local_path).to(device)
# print(f"NLLB model loaded successfully!")

# print(f"Model Loaded on : {nllb_model.device}")
# print(f"Model Parameters : {sum([p.numel() for p in nllb_model.parameters() if p.requires_grad is True])}")

In [6]:
from transformers import NllbTokenizer, AutoModelForSeq2SeqLM
from huggingface_hub import snapshot_download

# Download model locally first to bypass transformers/huggingface_hub version mismatch
nllb_local_path = snapshot_download("facebook/nllb-200-distilled-600M")
print(f"NLLB model downloaded to: {nllb_local_path}")

nllb_tokenizer = NllbTokenizer.from_pretrained(nllb_local_path)
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(nllb_local_path).to(device)
print(f"NLLB model loaded successfully!")

print(f"Model Loaded on : {nllb_model.device}")
print(f"Model Parameters : {sum([p.numel() for p in nllb_model.parameters() if p.requires_grad is True])}")

Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 9037.28it/s]


NLLB model downloaded to: C:\Users\Lenovo\.cache\huggingface\hub\models--facebook--nllb-200-distilled-600M\snapshots\f8d333a098d19b4fd9a8b18f94170487ad3f821d
NLLB model loaded successfully!
Model Loaded on : cuda:0
Model Parameters : 615073792


In [7]:
tes = ['Dilaporkan dua orang terluka cukup serius, sementara sisanya sudah diperbolehkan pulang setelah mendapatkan perawatan. Video gajah yang mengamuk itu kontan viral di media sosial. Si gajah berlari tak tentu arah, menabrak serta menginjak sebagian peserta upacara. Simak juga: Belum diketahui penyebab gajah tersebut tiba-tiba mengamuk. Diduga gajah itu kaget oleh sesuatu di antara para peserta dan pengunjung. Media setempat melaporkan gajah lain yang juga mengamuk di prosesi berbeda. Gajah hias merupakan daya tarik tersendiri dalam upacara keagamaan di Sri Lanka. Bagi warga Sri Lanka, memiliki gajah adalah simbol status. Beberapa kuil di Sri Lanka juga memiliki gajah untuk keperluan upacara.', 'Proyek MRT Sudirman- Lebak Bulus tengah dikerjakan, namun untuk koridor Timur -Barat dicabut dari PSN. Pembangunan prasarana didengungkan sebagai ujung tombak pemerintahan Presiden Joko Widodo dengan sasaran 245 proyek hingga tahun 2019 mendatang namun setahun sebelum batas waktu itu, 14 di antaranya dicoret dari daftar Proyek Strategis Nasional dengan alasan pengerjaan fisiknya tidak dapat dimulai sebelum kuartal III-2019. Pemangkasan proyek tersebut dilakukan setelah adanya evaluasi KPPIP di Kementerian Koordinator Bidang Perekonomian. Ketua Tim Pelaksana KPPIP Wahyu Utomo mengatakan berdasarkan evaluasi proyek tersebut tidak dapat memenuhi kriteria. "Nah kriteria yang kita pakai adalah utamanya kita minta proyek itu bisa dilakukan dimulai kontruksinya paling lambat sebelum kuartal III - 2019, jadi proyek yang tak bisa pekerjaan fisiknya sebelum kuartal III - 2019 kita usulkan statusnya kita lepas dulu sebagai PSN," jelas Wahyu. Empat belas proyek yang dikeluarkan dari daftar PSN ini memiliki nilai investasi sebesar Rp264 triliun. Wahyu menegaskan pencabutan proyek dari PSN itu bukan karena masalah anggaran. "Tidak, coba Anda lihat datanya ya anggaran untuk infrastukrur itu makin meningkat setiap tahunnya, jadi tak ada hubungannya alasan bahwa karena tidak ada dana segala macam. "Tapi upaya untuk mempercepatnya kita melibatkan swasta kita dorong, dari awal kita begitu niatnya. Uang APBN ini kita gunakan sebagai pendamping atau Viability Gap Funding untuk proyek kerja sama pemerintah swasta KPS," jelas dia. Sejumlah proyek yang dicabut dari PSN antara lain Kawasan Ekonomi Khusus (KEK) Merauke, MRT di Jakarta dan juga pembangunan jalan kereta api di wilayah Sumatera Selatan serta bandara di pulau Sebatik. "Kajian Kementerian Perhubungan tidak dibutuhkan segera karena arus penumpang masih dapat ditampung di Bandara Nunukan yang letaknya berdekatan," kata Wahyu. Jokowi ketika memantau proyek Mass Rapid Transport (MRT) di Jakarta. Sementara proyek lain yang melibatkan anggaran dari donor, masih harus menyelesaikan syarat. "Masalah eksternal, rencana mau dikasih pinjaman dari luar negeri sudah masuk tapi dari negara donor itu mensyaratkan ada kajian dulu, jadi ini kan jadi persyaratan untuk turunnya loan," jelas Wahyu. Sementara itu, untuk pembangunan kereta api di wilayah Sumatera Selatan akan tetap dilakukan namun menunggu kajian proyek selesai. Kabag Humas Pemerintah Provinsi Sumatera Selatan M Iqbal Alisyahbana mengatakan pencabutan proyek pembangunan sejumlah jalan kereta melintasi wilayahnya akan berdampak pada perekonomian daerahnya. "Mau bagaimana lagi, mungkin pemerintah ada proyek lain yang lebih diprioritasnya, meski sebenarnya sangat bagus bagi perekonomian Sumsel dibangun tepat waktu ya," jelas Iqbal. Evaluasi penting agar tak sia-sia Pengamat kebijakan publik Agus Pambagyo menyebutkan evaluasi yang dilakukan oleh pemerintah terhadap proyek startegis penting dilakukan agar proyek yang dijalankan dapat berdampak pada ekonomi masyarakat dan juga tidak sia-sia. Dia menyoroti pembangunan kereta bandara dan juga sky train yang tidak dirasakan manfaatnya. "Kebijakan membangun rel untuk kereta waktu itu setelah tujuh atau delapan tahun tidak ada kabarnya jadi dipakai oleh KRL. "Kalau sekarang dibagi dua itu ada problem bagi KRL karena harus dikurangi, lalu ada perubahan desain dari sky train itu tadinya di dalam, jadi yang naik sky train sudah pegang boarding pass," jelas Agus. "Apa bedanya sky train dengan bus Damri, itu yang saya tanyakan tapi ya dikerjakan terus". Stasiun kereta bandara yang berada di Dukuh Atas, menurut Agus, juga membuat warga enggan untuk naik kereta terutama yang tinggal jauh dari pusat kota. Kecelakaan kerja terjadi di proyek tol Becakayu pada Februari lalu, melukai tujuh pekerja. Cegah proyek mangkrak Wahyu menjelaskan evaluasi dilakukan pemerintah untuk memastikan proyek lain yang dilakukan dapat dimulai pembangunannya sebelum pemerintahan berakhir pada akhir tahun ini. Apakah ini dilakukan untuk mencegah proyek mangkrak? "Bisa dbilang begitu, karena kita tidak mau bicara proyek mangkrak ya, tapi kalau proyek itu sudah mulai itu akan lebih cenderung untuk terus dikerjakan. Itu saja sih. Kita tak ngomong begitu, karena proyek yang sudah dilepaskan statusnya dari PSN masih terus dikerjakan karena tidak dikeluarkan dari RPJMN," jelas Wahyu. Namun meski tidak lagi tercantum dalam PSN, menurut Wahyu, proyek-proyek tersebut masih termasuk dalam Rencana Pembangunan Jangka Menengah Nasional (RPJMN). Menurut Agus, evaluasi merupakan jalan yang terbaik untuk menghindari proyek mangkrak, karena Proyek Strategis Nasional dikerjakan berdasarkan peraturan presiden, sehingga sudah ditetapkan pelaksanaannya. "Semua proyek biasanya pakai perpres karena percepatan, dari pada melanggar perpres jadi sebaiknya dibatalkan," kata Agus. Transparansi perencanaan Direktur Institute for Development of Economics and Finance (INDEF) Enny Sri Hartati mengatakan penentuan dan pemangkasan proyek dalam PSN merupakan wewenang pemerintah, namun yang penting keputusan tersebut harus dilakukan secara transparan. "Yang utama adalah menurut saya, adalah mengapa 14 proyek itu dtunda, penundaan itu apakah disebabkan pemerintah mengalihkan itu," ujar Enny. Enny mengatakan yang harus dijelaskan misalnya dalam evaluasi ternyata proyek jalur kereta api tidak dibutuhkan di daerah tersebut, namun pemerintah menggantikanya dengan pembangunan yang mendesak seperti penyediaan air bersih. Dengan pemangkas proyek dalam PSN, namun tetap mencantumkannya dalam RPJMN, menurut Enny, menunjukkan adanya tumpang tindih perencanaan. Di sisi lain, ada proyek yang tidak ada dalam RPJMN namun masuk dalam PSN, seperti kereta api cepat. Padahal, dia mengatakan transparansi dokumen pembangunan ini penting untuk dijadikan acuan bagi dunia usaha untuk berinvestasi. "Di dalam benak pelaku dunia usaha, dengan perencanaan yang sudah sematang itu saja, ternyata ketika ada perubahan rezim atau pemerintahan bisa berubah. Pertanyaannya ketika ada perencanaan pembangunan yang dokumentasinya tidak jelas itu bagaimana kesinambungan dari proyek-proyek itu, itu mungkin salah satu yang menjadi pertanyaan besar dan menjadikan mengapa komitmen berbagai komitmen percepatan infrastruktur ini tak kunjung membuat pelaku usaha confident untuk melakukan investasi," jelas dia. Enny juga mencontohkan Progam Master Plan Percepatan dan Perluasan Pembangunan Ekonomi Indonesia (MP3I) yang memiliki dokumen lengkap dan bisa diakses pihak yang berkepentingan, seperti para pengusaha. Enny mengatakan proyek insfrastuktur sebaiknya bersifat jangka panjang dan tidak berhenti pada suatu rezim atau pemerintahan.']
# tes = [tes[1]]
inputs = nllb_tokenizer(tes, return_tensors="pt", max_length=None, padding=True, truncation=True).to(nllb_model.device)
tes = nllb_model.generate(
    **inputs, forced_bos_token_id = nllb_tokenizer.convert_tokens_to_ids("eng_Latn")
)
nllb_tokenizer.batch_decode(tes, skip_special_tokens=True)

c:\D\ITB\riset_gnn\generate_amr\venv\lib\site-packages\transformers\generation_utils.py:1202: UserWarning: Neither `max_length` nor `max_new_tokens` have been set, `max_length` will default to 200 (`self.config.max_length`). Controlling `max_length` via the config is deprecated and `max_length` will be removed from the config in v5 of Transformers -- we recommend using `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


['The video of the gajah who was killed went viral on social media. Si gajah berlari tak tentu arah, menabrak serta menginjak sebagian peserta upacara. Simak also: The cause of the gajah was not known. Diduga gajah was surprised by something between the participants and visitors. Local media reported that other gajah who were also killed in different proceedings. Gajah hias is a major attraction in the keagamaan ceremony in Sri Lanka. For Sri Lankans, gajah has symbolic status. Some temples in Sri Lanka also have gajah for ceremonial purposes.',
 'The development of the airport was announced as the final project of the government of President Joko Widodo with a target of 245 projects until 2019 but a year before the deadline, 14 of the projects were removed from the list of National Strategic Projects with the reason for the physical implementation cannot be started before the third quarter of 2019. The implementation of the project after the evaluation of KPPIP in the Ministry of Econ

# 3. Load Dataset (From CSV or jsonl)

In [8]:
def translate(texts):
    inputs = nllb_tokenizer(texts,
                            return_tensors="pt",
                            max_length = None,
                            padding = True,
                            truncation = True,
                            ).to(nllb_model.device)

    translated_tokens = nllb_model.generate(
        **inputs, forced_bos_token_id = nllb_tokenizer.convert_tokens_to_ids("eng_Latn")
    )

    return nllb_tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)

In [9]:
def load_dataset(dataset : str, from_csv : bool):
    pass

# 4. Create Custom Dataset & Custom DataLoader
This is needed for batch processing also keep track ID of dataset

In [10]:
XLSUM_DATASET_PATH = os.path.join("..", "data", "xlsum", "analysis_data.csv")
# TODO do parser for liputan6 dataset
# LIPUTAN6_DATASET_PATH =

class XLSumDataset(Dataset):
    """Custom Dataset For XLSum Only"""

    def __init__(self, path):
        self.df = pd.read_csv(path)
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        return {
            # here id is use so we don't lose track the data and can be used
            # as name of the file if we want to store all the amr graph on .txt
            "id" : row["id"],
            "text" : row["text"],
        }

In [11]:
def add_amr_mask(tokenized_inputs, masks):
    """
    Appends <AMR> <mask> </AMR> special tokens to each tokenized input,
    and extends the attention masks accordingly.

    Args:
        tokenized_inputs: List[List[int]] - batch of input_ids
        masks:            List[List[int]] - batch of attention_masks (1s and 0s)

    Returns:
        (updated_inputs, updated_masks): both with AMR tokens appended before padding
    """
    amr_suffix = [
        amr_tokenizer.amr_bos_token_id,
        amr_tokenizer.mask_token_id,
        amr_tokenizer.amr_eos_token_id,
    ]
    amr_mask_suffix = [1, 1, 1]  # these tokens should always be attended to

    updated_inputs = []
    updated_masks = []

    for input_ids, mask in zip(tokenized_inputs, masks):
        # Find where real tokens end and padding begins
        # (padding tokens have mask == 0)
        pad_start = mask.index(0) if 0 in mask else len(mask)

        # Insert AMR tokens before the padding
        new_input_ids = input_ids[:pad_start] + amr_suffix + input_ids[pad_start:]
        new_mask     = mask[:pad_start]      + amr_mask_suffix + mask[pad_start:]

        updated_inputs.append(new_input_ids)
        updated_masks.append(new_mask)

    return updated_inputs, updated_masks


def wrapper_collate_fn(prefix_lang1, prefix_lang2):
    def xlsum_collate_fn(batch):
        all_ids, all_texts = [], []
        
        for i in batch:
            all_ids.append(i["id"])
            all_texts.append(i["text"])

        # do translation and concat with lang_prefix and <AMR> <mask> </AMR>
        translated_texts = translate(all_texts)

        all_inputs = []
        for text, translated_text in zip(all_texts, translated_texts):
            cur_result = f"{prefix_lang1} {text} {prefix_lang2} {translated_text}"
            all_inputs.append(cur_result)

        tokenized_inputs = amr_tokenizer(
            all_inputs, max_length = None, padding = True, truncation = True
        )

        updated_inputs, updated_masks = add_amr_mask(
            tokenized_inputs["input_ids"],
            tokenized_inputs["attention_mask"]
        )

        return all_ids, updated_inputs, updated_masks

    return xlsum_collate_fn

    
    

In [12]:
ds = XLSumDataset(XLSUM_DATASET_PATH)

loader = DataLoader(ds, 2, collate_fn=wrapper_collate_fn("id_ID", "en_XX"))

# 5. Parsing AMR and Store it On .txt Folder

In [13]:
def store_graph(ids, graphs):
    os.makedirs("xlsum_amr_nafkhan", exist_ok=True)

    pass

for id, inputs, masks in loader:
    tes = amr_model.generate(
        input_ids = torch.tensor(inputs, dtype=torch.long).to(device),
        attention_mask=torch.tensor(masks, dtype=torch.long).to(device),
        num_beams=5,
        decoder_start_token_id = amr_tokenizer.amr_bos_token_id
    )
    break

c:\D\ITB\riset_gnn\generate_amr\venv\lib\site-packages\transformers\generation_utils.py:1202: UserWarning: Neither `max_length` nor `max_new_tokens` have been set, `max_length` will default to 200 (`self.config.max_length`). Controlling `max_length` via the config is deprecated and `max_length` will be removed from the config in v5 of Transformers -- we recommend using `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
print(tes)